# Data Analyst Job Postings — Data Cleaning Workflow

**Goal:** Prepare the raw Glassdoor Data Analyst job postings dataset for a separate EDA and insights notebook.

This notebook focuses only on cleaning and light feature engineering. The analysis, charts, and final insights should live in `02_eda_and_insights.ipynb` so the project remains easy to review.

**Cleaning approach:** keep the logic close to the original workflow — inspect the dataset, identify hidden missing values, clean core text fields, parse numeric ranges, validate the output, then export a clean CSV.

## Workflow

1. Import libraries and load the raw dataset  
2. Inspect structure, missing values, and hidden “ghost values”  
3. Remove rows that cannot support salary analysis  
4. Standardize column names and remove low-value columns  
5. Clean company name, salary, size, revenue, seniority, and company age  
6. Validate the cleaned dataset  
7. Export `DataAnalyst_cleaned.csv` for the EDA notebook

## 1. Setup

Import the core Python libraries used for data cleaning. Visualization libraries are included only for quick inspection if needed; detailed charts belong in the EDA notebook.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

## 2. Load the Raw Dataset

The raw file is loaded as-is. At this stage, the goal is not to clean anything yet. First, inspect what the dataset looks like and confirm the starting row/column count.

In [2]:
# Load raw dataset
raw_path = 'DataAnalyst.csv'
df = pd.read_csv(raw_path)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
df.head()

Rows: 2,253
Columns: 16


,Unnamed: 0,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors,Easy Apply
0,0,"Data Analyst, Center on Immigration and Justic...",$37K-$66K (Glassdoor est.),Are you eager to roll up your sleeves and harn...,3.2,Vera Institute of Justice\n3.2,"New York, NY","New York, NY",201 to 500 employees,1961,Nonprofit Organization,Social Assistance,Non-Profit,$100 to $500 million (USD),-1,True
1,1,Quality Data Analyst,$37K-$66K (Glassdoor est.),Overview\n\nProvides analytical and technical ...,3.8,Visiting Nurse Service of New York\n3.8,"New York, NY","New York, NY",10000+ employees,1893,Nonprofit Organization,Health Care Services & Hospitals,Health Care,$2 to $5 billion (USD),-1,-1
2,2,"Senior Data Analyst, Insights & Analytics Team...",$37K-$66K (Glassdoor est.),We’re looking for a Senior Data Analyst who ha...,3.4,Squarespace\n3.4,"New York, NY","New York, NY",1001 to 5000 employees,2003,Company - Private,Internet,Information Technology,Unknown / Non-Applicable,GoDaddy,-1
3,3,Data Analyst,$37K-$66K (Glassdoor est.),Requisition NumberRR-0001939\nRemote:Yes\nWe c...,4.1,Celerity\n4.1,"New York, NY","McLean, VA",201 to 500 employees,2002,Subsidiary or Business Segment,IT Services,Information Technology,$50 to $100 million (USD),-1,-1
4,4,Reporting Data Analyst,$37K-$66K (Glassdoor est.),ABOUT FANDUEL GROUP\n\nFanDuel Group is a worl...,3.9,FanDuel\n3.9,"New York, NY","New York, NY",501 to 1000 employees,2009,Company - Private,Sports & Recreation,"Arts, Entertainment & Recreation",$100 to $500 million (USD),DraftKings,True


## 3. Initial Data Quality Check

`df.info()` and `isnull()` suggest the dataset is mostly complete. However, Glassdoor-style scraped datasets often use values like `-1`, `Unknown`, or `Unknown / Non-Applicable` instead of real nulls. Those are hidden missing values and must be handled deliberately.

In [3]:
# Basic structure and visible missing values
df.info()
print("\nVisible null values:")
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2253 entries, 0 to 2252
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         2253 non-null   int64  
 1   Job Title          2253 non-null   object 
 2   Salary Estimate    2253 non-null   object 
 3   Job Description    2253 non-null   object 
 4   Rating             2253 non-null   float64
 5   Company Name       2252 non-null   object 
 6   Location           2253 non-null   object 
 7   Headquarters       2253 non-null   object 
 8   Size               2253 non-null   object 
 9   Founded            2253 non-null   int64  
 10  Type of ownership  2253 non-null   object 
 11  Industry           2253 non-null   object 
 12  Sector             2253 non-null   object 
 13  Revenue            2253 non-null   object 
 14  Competitors        2253 non-null   object 
 15  Easy Apply         2253 non-null   object 
dtypes: float64(1), int64(2),

In [4]:
# Check the one true null in Company Name before broader cleaning
df[df['Company Name'].isnull()]

,Unnamed: 0,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors,Easy Apply
1860,1860,Data Analyst,$53K-$99K (Glassdoor est.),"Kindred at Home, part of the Kindred at Home f...",-1.0,NaN,"Mooresville, NC",-1,-1,-1,-1,-1,-1,-1,-1,-1


### Decision: Drop the row with missing company name

The row with a missing company name does not contain enough useful information for the project. Since it is only one row, removing it will not materially affect the analysis.

In [5]:
# Drop rows where Company Name is missing
before_rows = len(df)
df = df[df['Company Name'].notna()].copy()

print(f"Rows removed: {before_rows - len(df)}")
print(f"Remaining rows: {len(df):,}")

Rows removed: 1
Remaining rows: 2,252


## 4. Standardize Column Names

The original column names are readable but inconsistent for coding. I kept the beginner-readable style from the original notebook: lowercase column names with spaces. This avoids making the cleaning notebook feel disconnected from the EDA notebook.

In [6]:
# Standardize column titles for easier navigation
df.columns = df.columns.str.strip().str.lower()
df.columns.tolist()

['unnamed: 0',
 'job title',
 'salary estimate',
 'job description',
 'rating',
 'company name',
 'location',
 'headquarters',
 'size',
 'founded',
 'type of ownership',
 'industry',
 'sector',
 'revenue',
 'competitors',
 'easy apply']

## 5. Audit Hidden Missing Values

The next step is to inspect object/text columns and numeric boundary values. This confirms which values are actually missing but disguised as regular data.

In [7]:
# Inspect common values in text columns
text_columns = df.select_dtypes(include='object').columns

for col in text_columns:
    print(f"\nColumn: {col}")
    print(df[col].value_counts(dropna=False).head(10))
    print("-" * 80)


Column: job title
job title
Data Analyst               404
Senior Data Analyst         90
Junior Data Analyst         30
Business Data Analyst       28
Sr. Data Analyst            21
Data Analyst Junior         17
Data Analyst II             17
Data Quality Analyst        17
Data Governance Analyst     16
Lead Data Analyst           15
Name: count, dtype: int64
--------------------------------------------------------------------------------

Column: salary estimate
salary estimate
$41K-$78K (Glassdoor est.)     57
$42K-$76K (Glassdoor est.)     57
$50K-$86K (Glassdoor est.)     41
$35K-$67K (Glassdoor est.)     33
$60K-$124K (Glassdoor est.)    31
$58K-$93K (Glassdoor est.)     31
$43K-$76K (Glassdoor est.)     31
$46K-$87K (Glassdoor est.)     30
$51K-$87K (Glassdoor est.)     30
$37K-$66K (Glassdoor est.)     30
Name: count, dtype: int64
--------------------------------------------------------------------------------

Column: job description
job description
Are you eager to roll up 

In [8]:
# Inspect numeric columns for impossible values
numeric_cols = ['rating', 'founded']

for col in numeric_cols:
    print(f"Column: {col}")
    print(f"Minimum value: {df[col].min()}")
    print(f"Maximum value: {df[col].max()}")
    print(f"Impossible values (<= 0): {(df[col] <= 0).sum()}")
    print("-" * 80)

Column: rating
Minimum value: -1.0
Maximum value: 5.0
Impossible values (<= 0): 271
--------------------------------------------------------------------------------
Column: founded
Minimum value: -1
Maximum value: 2019
Impossible values (<= 0): 659
--------------------------------------------------------------------------------


### Decision: Convert ghost values into real missing values

Values like `-1`, `Unknown`, and `Unknown / Non-Applicable` are not valid analytical categories. They represent missing or undisclosed information, so they should be converted to `NaN`.

In [9]:
ghost_values = ['-1', '-1.0', -1, -1.0, 'Unknown', 'Unknown / Non-Applicable']

df.replace(ghost_values, np.nan, inplace=True)

print("Null values after replacing ghost values:")
print(df.isnull().sum().sort_values(ascending=False))

Null values after replacing ghost values:
easy apply           2172
competitors          1731
revenue               777
founded               659
sector                352
industry              352
rating                271
size                  204
type of ownership     178
headquarters          171
salary estimate         1
location                0
unnamed: 0              0
company name            0
job title               0
job description         0
dtype: int64


## 6. Remove Rows and Columns That Do Not Support the Analysis

The project is salary-focused, so rows without `salary estimate` cannot support the main analysis. Some columns also add little value: `unnamed: 0` is an index artifact, while `easy apply` and `competitors` are mostly missing.

In [10]:
# Drop rows with missing salary estimate because salary is the core metric
before_rows = len(df)
df = df[df['salary estimate'].notna()].copy()

print(f"Rows removed due to missing salary estimate: {before_rows - len(df)}")
print(f"Remaining rows: {len(df):,}")

Rows removed due to missing salary estimate: 1
Remaining rows: 2,251


In [11]:
# Drop low-value columns
columns_to_drop = ['unnamed: 0', 'easy apply', 'competitors']
df.drop(columns=columns_to_drop, inplace=True)

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2251 entries, 0 to 2252
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   job title          2251 non-null   object 
 1   salary estimate    2251 non-null   object 
 2   job description    2251 non-null   object 
 3   rating             1980 non-null   float64
 4   company name       2251 non-null   object 
 5   location           2251 non-null   object 
 6   headquarters       2080 non-null   object 
 7   size               2047 non-null   object 
 8   founded            1592 non-null   float64
 9   type of ownership  2073 non-null   object 
 10  industry           1899 non-null   object 
 11  sector             1899 non-null   object 
 12  revenue            1474 non-null   object 
dtypes: float64(2), object(11)
memory usage: 246.2+ KB


## 7. Missing Value Decisions for Remaining Columns

Not every missing value should be dropped or imputed. These are the cleaning decisions used before EDA:

| Column | Decision | Reason |
|---|---|---|
| `rating` | Keep as null | Some companies may not have enough Glassdoor reviews |
| `headquarters` and `size` | Keep as null | Missing company profile details should not remove job postings |
| `founded` and `type of ownership` | Keep as null | No safe way to infer these values |
| `industry` and `sector` | Keep as null for now | These can be enriched in a separate feature engineering or EDA step |
| `revenue` | Label as `Not disclosed` | Useful as a category, but numeric parsing remains null when undisclosed |

In [12]:
# Keep missing revenue as a transparent category for categorical analysis
df['revenue'] = df['revenue'].fillna('Not disclosed')

# Ensure rating and founded use appropriate numeric types
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df['founded'] = df['founded'].astype('Int64')

print(df[['rating', 'founded', 'revenue']].dtypes)
print("\nRevenue categories:")
print(df['revenue'].value_counts(dropna=False))

rating     float64
founded      Int64
revenue     object
dtype: object

Revenue categories:
revenue
Not disclosed                       777
$100 to $500 million (USD)          218
$50 to $100 million (USD)           199
$10+ billion (USD)                  189
$10 to $25 million (USD)            132
$2 to $5 billion (USD)              129
$1 to $5 million (USD)              111
$25 to $50 million (USD)            109
Less than $1 million (USD)           93
$1 to $2 billion (USD)               87
$500 million to $1 billion (USD)     79
$5 to $10 million (USD)              72
$5 to $10 billion (USD)              56
Name: count, dtype: int64


## 8. Clean Company Name

The raw company name contains the company rating after a newline, such as `Company Name
3.9`. The rating already exists as a separate column, so the company name should be cleaned to contain only the name.

In [13]:
# Remove rating suffix from company name and trim whitespace
df['company name'] = df['company name'].str.split('\n').str[0].str.strip()

print(df[['company name', 'rating']].head(10))

                         company name  rating
0           Vera Institute of Justice     3.2
1  Visiting Nurse Service of New York     3.8
2                         Squarespace     3.4
3                            Celerity     4.1
4                             FanDuel     3.9
5                             Point72     3.9
6                           Two Sigma     4.4
7             GNY Insurance Companies     3.7
8                                DMGT     4.0
9                           Riskified     4.4


## 9. Parse Salary Range

The raw salary is stored as text, for example `$41K-$78K (Glassdoor est.)`. This needs to be converted into numeric columns so the EDA notebook can compare salaries across sector, location, company size, and seniority.

In [14]:
# Inspect salary patterns before parsing
print(f"Unique salary patterns: {df['salary estimate'].nunique()}")
print(df['salary estimate'].value_counts().head(10))

Unique salary patterns: 89
salary estimate
$41K-$78K (Glassdoor est.)     57
$42K-$76K (Glassdoor est.)     57
$50K-$86K (Glassdoor est.)     41
$35K-$67K (Glassdoor est.)     33
$43K-$76K (Glassdoor est.)     31
$60K-$124K (Glassdoor est.)    31
$58K-$93K (Glassdoor est.)     31
$46K-$87K (Glassdoor est.)     30
$37K-$66K (Glassdoor est.)     30
$43K-$69K (Glassdoor est.)     30
Name: count, dtype: int64


In [15]:
# Clean salary text and split into min/max
salary_clean = df['salary estimate'].str.replace('$', '', regex=False)
salary_clean = salary_clean.str.replace('K', '', regex=False)
salary_clean = salary_clean.str.replace(r'\(.*\)', '', regex=True).str.strip()

salary_split = salary_clean.str.split('-', expand=True)

df['salary_min'] = pd.to_numeric(salary_split[0], errors='coerce') * 1000
df['salary_max'] = pd.to_numeric(salary_split[1], errors='coerce') * 1000
df['salary_avg'] = ((df['salary_min'] + df['salary_max']) / 2).round(0)

# Use nullable integer type so missing values can still exist if parsing fails
for col in ['salary_min', 'salary_max', 'salary_avg']:
    df[col] = df[col].astype('Int64')

print(df[['salary estimate', 'salary_min', 'salary_max', 'salary_avg']].head(10))

              salary estimate  salary_min  salary_max  salary_avg
0  $37K-$66K (Glassdoor est.)       37000       66000       51500
1  $37K-$66K (Glassdoor est.)       37000       66000       51500
2  $37K-$66K (Glassdoor est.)       37000       66000       51500
3  $37K-$66K (Glassdoor est.)       37000       66000       51500
4  $37K-$66K (Glassdoor est.)       37000       66000       51500
5  $37K-$66K (Glassdoor est.)       37000       66000       51500
6  $37K-$66K (Glassdoor est.)       37000       66000       51500
7  $37K-$66K (Glassdoor est.)       37000       66000       51500
8  $37K-$66K (Glassdoor est.)       37000       66000       51500
9  $37K-$66K (Glassdoor est.)       37000       66000       51500


## 10. Parse Company Size Range

The `size` column is stored as text ranges like `51 to 200 employees`. For EDA, numeric company size fields make it easier to group or compare salary patterns by company size.

In [16]:
# Inspect size categories
print(df['size'].value_counts(dropna=False))

size
51 to 200 employees        420
10000+ employees           375
1001 to 5000 employees     348
1 to 50 employees          347
201 to 500 employees       249
501 to 1000 employees      211
NaN                        204
5001 to 10000 employees     97
Name: count, dtype: int64


In [17]:
# Clean size text and split range values
size_clean = df['size'].str.replace(' employees', '', case=False, regex=False)
size_clean = size_clean.str.strip()

# Open-ended values like 10000+ have a known minimum but unknown maximum
is_open_ended_size = size_clean.str.contains(r'\+', na=False)
size_clean = size_clean.str.replace('+', '', regex=False)

size_split = size_clean.str.split(' to ', expand=True)

df['size_min'] = pd.to_numeric(size_split[0], errors='coerce')
df['size_max'] = pd.to_numeric(size_split[1], errors='coerce')

# Keep max and average blank for open-ended ranges to avoid inventing a fake ceiling
df.loc[is_open_ended_size, 'size_max'] = np.nan
df['size_avg'] = (df['size_min'] + df['size_max']) / 2

for col in ['size_min', 'size_max', 'size_avg']:
    df[col] = df[col].round(0).astype('Int64')

print(df[['size', 'size_min', 'size_max', 'size_avg']].drop_duplicates().sort_values('size_min'))

                       size  size_min  size_max  size_avg
11        1 to 50 employees         1        50        26
17      51 to 200 employees        51       200       126
0      201 to 500 employees       201       500       350
4     501 to 1000 employees       501      1000       750
2    1001 to 5000 employees      1001      5000      3000
8   5001 to 10000 employees      5001     10000      7500
1          10000+ employees     10000      <NA>      <NA>
21                      NaN      <NA>      <NA>      <NA>


## 11. Parse Revenue Range

Revenue has more edge cases than salary and size because it mixes millions, billions, open-ended values, and `Less than $1 million`. A small helper function is used here because this logic is easier to read and validate than forcing one long chained expression.

In [18]:
# Inspect revenue categories
print(df['revenue'].value_counts(dropna=False))

revenue
Not disclosed                       777
$100 to $500 million (USD)          218
$50 to $100 million (USD)           199
$10+ billion (USD)                  189
$10 to $25 million (USD)            132
$2 to $5 billion (USD)              129
$1 to $5 million (USD)              111
$25 to $50 million (USD)            109
Less than $1 million (USD)           93
$1 to $2 billion (USD)               87
$500 million to $1 billion (USD)     79
$5 to $10 million (USD)              72
$5 to $10 billion (USD)              56
Name: count, dtype: int64


In [19]:
def revenue_value_to_millions(text_part, fallback_unit=None):
    '''Convert one side of a revenue range into millions of USD.'''
    text_part = str(text_part).lower().replace('$', '').replace('(usd)', '').strip()
    number_match = re.search(r'(\d+(?:\.\d+)?)', text_part)

    if number_match is None:
        return np.nan

    value = float(number_match.group(1))

    if 'billion' in text_part:
        return value * 1000
    if 'million' in text_part:
        return value
    if fallback_unit == 'billion':
        return value * 1000
    if fallback_unit == 'million':
        return value

    return np.nan


def parse_revenue_to_millions(revenue_text):
    '''Return revenue_min, revenue_max, revenue_avg in millions of USD.'''
    if pd.isna(revenue_text) or revenue_text == 'Not disclosed':
        return pd.Series([np.nan, np.nan, np.nan])

    text = str(revenue_text).lower().strip()

    if 'less than' in text and 'million' in text:
        return pd.Series([0.0, 1.0, 0.5])

    # Open-ended values such as $10+ billion have a known minimum but no safe maximum.
    if '+' in text:
        revenue_min = revenue_value_to_millions(text)
        return pd.Series([revenue_min, np.nan, np.nan])

    if ' to ' in text:
        left, right = text.split(' to ', 1)

        # If the left side has no unit, borrow the right-side unit.
        fallback_unit = 'billion' if 'billion' in right else 'million' if 'million' in right else None

        revenue_min = revenue_value_to_millions(left, fallback_unit=fallback_unit)
        revenue_max = revenue_value_to_millions(right)
        revenue_avg = (revenue_min + revenue_max) / 2 if pd.notna(revenue_min) and pd.notna(revenue_max) else np.nan
        return pd.Series([revenue_min, revenue_max, revenue_avg])

    return pd.Series([np.nan, np.nan, np.nan])


df[['revenue_min_millions', 'revenue_max_millions', 'revenue_avg_millions']] = df['revenue'].apply(parse_revenue_to_millions)

print(df[['revenue', 'revenue_min_millions', 'revenue_max_millions', 'revenue_avg_millions']].drop_duplicates())

                              revenue  ...  revenue_avg_millions
0          $100 to $500 million (USD)  ...                 300.0
1              $2 to $5 billion (USD)  ...                3500.0
2                       Not disclosed  ...                   NaN
3           $50 to $100 million (USD)  ...                  75.0
8              $1 to $2 billion (USD)  ...                1500.0
10            $5 to $10 billion (USD)  ...                7500.0
14             $1 to $5 million (USD)  ...                   3.0
17           $25 to $50 million (USD)  ...                  37.5
20                 $10+ billion (USD)  ...                   NaN
25         Less than $1 million (USD)  ...                   0.5
26           $10 to $25 million (USD)  ...                  17.5
45   $500 million to $1 billion (USD)  ...                 750.0
106           $5 to $10 million (USD)  ...                   7.5

[13 rows x 4 columns]

## 12. Add Seniority Level

Seniority is extracted from the job title using keyword logic. This is not perfect, but it creates a practical grouping for EDA without overcomplicating the cleaning notebook.

In [20]:
seniority = []

for title in df['job title']:
    title_lower = str(title).lower()

    if any(word in title_lower for word in ['senior', 'sr', 'lead', 'principal', 'director', 'manager']):
        seniority.append('senior')
    elif any(word in title_lower for word in ['junior', 'jr', 'entry', 'associate', 'intern']):
        seniority.append('entry')
    else:
        seniority.append('mid/unspecified')

df['seniority_level'] = seniority

print(df[['job title', 'seniority_level']].drop_duplicates().head(20))
print("\nSeniority distribution:")
print(df['seniority_level'].value_counts())

                                            job title  seniority_level
0   Data Analyst, Center on Immigration and Justic...  mid/unspecified
1                                Quality Data Analyst  mid/unspecified
2   Senior Data Analyst, Insights & Analytics Team...           senior
3                                        Data Analyst  mid/unspecified
4                              Reporting Data Analyst  mid/unspecified
6                        Business/Data Analyst (FP&A)  mid/unspecified
7                                Data Science Analyst  mid/unspecified
9                       Data Analyst, Merchant Health  mid/unspecified
12                                       DATA ANALYST  mid/unspecified
13                                Senior Data Analyst           senior
14                   Investment Advisory Data Analyst  mid/unspecified
15                        Sustainability Data Analyst  mid/unspecified
17                              Clinical Data Analyst  mid/unspecified
18    

## 13. Add Company Age

`company_age` is calculated from the `founded` year. I used a fixed analysis year so the notebook stays reproducible when rerun later.

In [21]:
analysis_year = 2026

df['company_age'] = analysis_year - df['founded']
df.loc[df['founded'].isna(), 'company_age'] = pd.NA
df['company_age'] = df['company_age'].astype('Int64')

print(df[['company name', 'founded', 'company_age']].head(10))

                         company name  founded  company_age
0           Vera Institute of Justice     1961           65
1  Visiting Nurse Service of New York     1893          133
2                         Squarespace     2003           23
3                            Celerity     2002           24
4                             FanDuel     2009           17
5                             Point72     2014           12
6                           Two Sigma     2001           25
7             GNY Insurance Companies     1914          112
8                                DMGT     1896          130
9                           Riskified     2013           13


## 14. Reorder Columns for Readability

The final dataset is arranged so the main job/company fields appear first, followed by engineered numeric fields. This makes the cleaned CSV easier to inspect and easier to use in the EDA notebook.

In [22]:
preferred_order = [
    'job title',
    'seniority_level',
    'salary estimate',
    'salary_min',
    'salary_max',
    'salary_avg',
    'job description',
    'rating',
    'company name',
    'location',
    'headquarters',
    'size',
    'size_min',
    'size_max',
    'size_avg',
    'founded',
    'company_age',
    'type of ownership',
    'industry',
    'sector',
    'revenue',
    'revenue_min_millions',
    'revenue_max_millions',
    'revenue_avg_millions'
]

# Keep only columns that exist, then append any unexpected remaining columns at the end.
ordered_columns = [col for col in preferred_order if col in df.columns]
remaining_columns = [col for col in df.columns if col not in ordered_columns]
df = df[ordered_columns + remaining_columns]

df.head()

,job title,seniority_level,salary estimate,salary_min,salary_max,salary_avg,job description,rating,company name,location,headquarters,size,size_min,size_max,size_avg,founded,company_age,type of ownership,industry,sector,revenue,revenue_min_millions,revenue_max_millions,revenue_avg_millions
0,"Data Analyst, Center on Immigration and Justic...",mid/unspecified,$37K-$66K (Glassdoor est.),37000,66000,51500,Are you eager to roll up your sleeves and harn...,3.2,Vera Institute of Justice,"New York, NY","New York, NY",201 to 500 employees,201,500,350,1961,65,Nonprofit Organization,Social Assistance,Non-Profit,$100 to $500 million (USD),100.0,500.0,300.0
1,Quality Data Analyst,mid/unspecified,$37K-$66K (Glassdoor est.),37000,66000,51500,Overview\n\nProvides analytical and technical ...,3.8,Visiting Nurse Service of New York,"New York, NY","New York, NY",10000+ employees,10000,<NA>,<NA>,1893,133,Nonprofit Organization,Health Care Services & Hospitals,Health Care,$2 to $5 billion (USD),2000.0,5000.0,3500.0
2,"Senior Data Analyst, Insights & Analytics Team...",senior,$37K-$66K (Glassdoor est.),37000,66000,51500,We’re looking for a Senior Data Analyst who ha...,3.4,Squarespace,"New York, NY","New York, NY",1001 to 5000 employees,1001,5000,3000,2003,23,Company - Private,Internet,Information Technology,Not disclosed,NaN,NaN,NaN
3,Data Analyst,mid/unspecified,$37K-$66K (Glassdoor est.),37000,66000,51500,Requisition NumberRR-0001939\nRemote:Yes\nWe c...,4.1,Celerity,"New York, NY","McLean, VA",201 to 500 employees,201,500,350,2002,24,Subsidiary or Business Segment,IT Services,Information Technology,$50 to $100 million (USD),50.0,100.0,75.0
4,Reporting Data Analyst,mid/unspecified,$37K-$66K (Glassdoor est.),37000,66000,51500,ABOUT FANDUEL GROUP\n\nFanDuel Group is a worl...,3.9,FanDuel,"New York, NY","New York, NY",501 to 1000 employees,501,1000,750,2009,17,Company - Private,Sports & Recreation,"Arts, Entertainment & Recreation",$100 to $500 million (USD),100.0,500.0,300.0


## 15. Final Validation

Before exporting, validate the final row count, schema, remaining nulls, and parsing results. The purpose is not to eliminate every null, but to make sure missing values are intentional and documented.

In [23]:
print(f"Final rows: {df.shape[0]:,}")
print(f"Final columns: {df.shape[1]:,}")

print("\nFinal data types:")
print(df.dtypes)

print("\nRemaining null values:")
print(df.isnull().sum().sort_values(ascending=False))

Final rows: 2,251
Final columns: 24

Final data types:
job title                object
seniority_level          object
salary estimate          object
salary_min                Int64
salary_max                Int64
salary_avg                Int64
job description          object
rating                  float64
company name             object
location                 object
headquarters             object
size                     object
size_min                  Int64
size_max                  Int64
size_avg                  Int64
founded                   Int64
company_age               Int64
type of ownership        object
industry                 object
sector                   object
revenue                  object
revenue_min_millions    float64
revenue_max_millions    float64
revenue_avg_millions    float64
dtype: object

Remaining null values:


revenue_max_millions    966
revenue_avg_millions    966
revenue_min_millions    777
founded                 659
company_age             659
size_max                579
size_avg                579
industry                352
sector                  352
rating                  271
size_min                204
size                    204
type of ownership       178
headquarters            171
job title                 0
salary estimate           0
job description           0
salary_avg                0
salary_max                0
salary_min                0
seniority_level           0
company name              0
location                  0
revenue                   0
dtype: int64


In [24]:
# Validate that core salary fields parsed successfully
salary_check = df[['salary estimate', 'salary_min', 'salary_max', 'salary_avg']].isnull().sum()
print(salary_check)

# Quick descriptive check for salary columns
print("\nSalary summary:")
print(df[['salary_min', 'salary_max', 'salary_avg']].describe())

salary estimate    0
salary_min         0
salary_max         0
salary_avg         0
dtype: int64

Salary summary:
         salary_min    salary_max    salary_avg
count        2251.0        2251.0        2251.0
mean   54267.436695  89975.122168  72121.279431
std    19579.706082  29321.502211  23605.836291
min         24000.0       38000.0       33500.0
25%         41000.0       70000.0       58000.0
50%         50000.0       87000.0       69000.0
75%         64000.0      104000.0       80500.0
max        113000.0      190000.0      150000.0


In [25]:
# Check disclosed revenue values without an average.
# These are usually open-ended values like '$10+ billion (USD)', where a max/average should not be invented.
open_ended_revenue_check = df[(df['revenue'] != 'Not disclosed') & (df['revenue_avg_millions'].isna())]

print(f"Rows with disclosed revenue but no safe revenue average: {len(open_ended_revenue_check):,}")
print(open_ended_revenue_check['revenue'].value_counts(dropna=False))

Rows with disclosed revenue but no safe revenue average: 189
revenue
$10+ billion (USD)    189
Name: count, dtype: int64


## 16. Export Cleaned Dataset

The cleaned file becomes the input for the separate EDA and insights notebook.

In [26]:
output_path = 'DataAnalyst_cleaned.csv'
df.to_csv(output_path, index=False)

print(f"Cleaned dataset exported to: {output_path}")

Cleaned dataset exported to: DataAnalyst_cleaned.csv


## Cleaning Summary

The cleaned dataset is now ready for EDA. Key changes:

- Removed rows without enough company or salary information
- Converted hidden missing values into true nulls
- Dropped low-value columns
- Cleaned company names
- Parsed salary, size, and revenue ranges into numeric fields
- Added seniority level and company age
- Preserved nulls where there was no defensible way to impute values

The next notebook should focus on analysis questions, visuals, and insights — not repeating the full cleaning workflow.